# Append EDO Shapefiles and Export to GeoJSON

This notebook appends multiple dated European Drought Observatory (EDO) shapefiles into one temporary feature class and exports the result as a single GeoJSON file.

Each input shapefile represents one date and contains polygon or multipolygon features with category and value attributes. The date is already stored in the attribute table, so this workflow only combines the features without dissolving or altering the geometries.

> **Important:** This notebook requires the ArcGIS Pro Python environment because it uses `arcpy`.

## 1. Import required libraries

The workflow uses:

- `arcpy` for geoprocessing
- `Path` from `pathlib` for reliable file path handling
- `os` for general path and file utilities

In [ ]:
import arcpy
import os
from pathlib import Path

## 2. Define input and output paths

Set the input folder where the shapefiles are stored and the output folder where the final GeoJSON file will be saved.

Only shapefiles ending with `_diss.shp` will be selected later in the workflow.

In [ ]:
# Input folder containing the dated shapefiles
input_folder = Path(r"C:.\Your_input_folder")

# Output folder for the final GeoJSON
output_folder = Path(r"C:.\Your_output_folder")

# Output GeoJSON file name
output_geojson = output_folder / "edo_append.geojson"

# Allow overwriting existing outputs
arcpy.env.overwriteOutput = True

# Create the output folder if it does not already exist
output_folder.mkdir(parents=True, exist_ok=True)

print("Input folder:", input_folder)
print("Output GeoJSON:", output_geojson)

## 3. Select the input shapefiles

This cell searches the input folder and selects only files ending with `_diss.shp`.

The list is sorted to keep the processing order consistent.

In [ ]:
# Select only shapefiles ending with _diss.shp
shapefiles = sorted(input_folder.glob("*_diss.shp"))

print(f"Number of shapefiles found: {len(shapefiles)}")

if not shapefiles:
    raise Exception("No *_diss.shp files were found in the input folder.")

# Show the first few selected files as a quick check
for shp in shapefiles[:5]:
    print(shp.name)

## 4. Create a temporary feature class

ArcPy cannot append directly into a GeoJSON file. Therefore, the workflow first creates a temporary feature class in the ArcGIS scratch geodatabase.

The first shapefile is copied into this temporary feature class and used as the target dataset for the append process.

In [ ]:
# Use the ArcGIS scratch geodatabase for temporary processing
scratch_gdb = arcpy.env.scratchGDB
temp_fc = scratch_gdb + r"\merged_fc"

# Delete the temporary feature class if it already exists
if arcpy.Exists(temp_fc):
    arcpy.management.Delete(temp_fc)

# Copy the first shapefile to create the target feature class
arcpy.conversion.FeatureClassToFeatureClass(
    in_features=str(shapefiles[0]),
    out_path=scratch_gdb,
    out_name="merged_fc"
)

print("First shapefile copied to temporary feature class.")
print("Temporary feature class:", temp_fc)

## 5. Append the remaining shapefiles

Each remaining shapefile is appended to the temporary feature class.

The `NO_TEST` schema option is used so that ArcPy does not require the input schemas to be perfectly identical. Fields with matching names are preserved in the target feature class.

This step does **not** dissolve, merge, or modify geometries. It only stacks the features into one dataset.

In [ ]:
# Append all remaining shapefiles into the temporary feature class
for shp in shapefiles[1:]:
    print(f"Append: {shp.name}")

    arcpy.management.Append(
        inputs=str(shp),
        target=temp_fc,
        schema_type="NO_TEST"
    )

print("Append completed.")

## 6. Check the number of output features

This check helps confirm whether all features were appended successfully.

For example, if each shapefile contains 4 features and there are 80 shapefiles, the expected result is approximately 320 features.

In [ ]:
# Count features in the temporary feature class
count = int(arcpy.management.GetCount(temp_fc)[0])

print(f"Total number of features: {count}")
print(f"Number of input shapefiles: {len(shapefiles)}")

## 7. Export the appended layer to GeoJSON

The temporary feature class is exported to a single GeoJSON file.

If a GeoJSON file with the same name already exists, it is deleted first to avoid export errors.

In [ ]:
# Delete existing GeoJSON if it already exists
if output_geojson.exists():
    output_geojson.unlink()

# Export the temporary feature class to GeoJSON
arcpy.conversion.FeaturesToJSON(
    in_features=temp_fc,
    out_json_file=str(output_geojson),
    format_json="FORMATTED",
    geoJSON="GEOJSON"
)

print(f"GeoJSON successfully created: {output_geojson}")

## 8. Notes for using the GeoJSON

The exported GeoJSON contains all appended polygon or multipolygon features in one file.

It can be used for:

- time-enabled visualization in ArcGIS Pro, if the date field is correctly recognized
- web mapping workflows, such as Leaflet
- additional processing in GIS or Python environments

For time slider workflows, make sure the date attribute is stored in a valid date format or can be parsed by the target software.